In [1]:
import sys

import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)

3.12.10 2.3.4 2.3.3 1.7.2 3.10.7


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 3132
data = pd.read_csv("data/spambase.csv")
feature_names = [c for c in data.columns if c != "spam"]
X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)

(4204, 57) 39.9 (3153, 57) (1051, 57)


SEED = 3132. The split and the folds use it, and so does every random model.

In [3]:
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=SEED)
rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1)
lr = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=2000))
bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)
knn = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)
members = [("lr", lr), ("knn", knn), ("tree", tree)]   # members of voting and stacking in Parts 3-4

The models are defined once here and not changed later. When I fit one more than once I use `clone`.

## Part 1 — Bagging and out-of-bag estimates

In [4]:
def oob_mask(sample_indices, n_samples):
    """Boolean mask of the rows that do NOT occur in sample_indices."""
    mask = np.ones(n_samples, dtype=bool)
    mask[np.asarray(sample_indices, dtype=int)] = False
    return mask

In [5]:
print(oob_mask([0, 2, 2], 5))
print(oob_mask([], 3))
print(oob_mask(np.array([3, 0]), 4))

[False  True False  True  True]
[ True  True  True]
[False  True  True False]


In [6]:
bag.fit(X_train, y_train)
n = len(y_train)

oob_share = np.mean([oob_mask(s, n).mean() for s in bag.estimators_samples_])
print(f"mean OOB share:      {oob_share:.4f}")
print(f"mean in-bag share:   {1 - oob_share:.4f}")
print(f"(1 - 1/n)^n:         {(1 - 1/n) ** n:.4f}")
print(f"1 - (1 - 1/n)^n:     {1 - (1 - 1/n) ** n:.4f}")

mean OOB share:      0.3683
mean in-bag share:   0.6317
(1 - 1/n)^n:         0.3678
1 - (1 - 1/n)^n:     0.6322


In [7]:
votes = np.zeros((n, 2), dtype=int)
for est, samples in zip(bag.estimators_, bag.estimators_samples_):
    mask = oob_mask(samples, n)
    pred = est.predict(X_train[mask])
    votes[np.where(mask)[0], pred] += 1

oob_pred = votes.argmax(axis=1)      # tie goes to class 0
my_oob_acc = (oob_pred == y_train).mean()
print(f"my OOB accuracy:  {my_oob_acc:.4f}")
print(f"bag.oob_score_:   {bag.oob_score_:.4f}")

my OOB accuracy:  0.9369
bag.oob_score_:   0.9369


On average a tree misses 36.8% of the training rows (OOB) and sees 63.2% of them. The formula (1 - 1/n)^n gives 36.8% too, so the bootstrap behaves as expected: each tree has about a third of the rows that it never saw.

My own OOB accuracy is 0.9369 and `bag.oob_score_` is 0.9369. They are the same, so my vote counting does what scikit-learn does.